In [1]:
!pip -q install reportlab pdfplumber
!mkdir -p lablens
open("lablens/__init__.py", "w").write('"""LabLens: lab-report intelligence demo (extract -> trend -> doctor-ready summary). Synthetic data only."""\n')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 64.9 MB/s eta 0:00:00


109

In [2]:
%%writefile lablens/catalog.py
"""Test catalogue: canonical names, the aliases different labs print, units and reference ranges (male, female)."""
import re

# name: (aliases, unit, ref_male, ref_female, decimals, group)   ref = (low, high), None = open ended
TESTS = {
    "HbA1c": (["Hemoglobin A1c", "HbA1c", "Glycosylated Hemoglobin HbA1c"], "%", (4.0, 5.6), (4.0, 5.6), 1, "Glycemic"),
    "Fasting Glucose": (["Fasting Blood Sugar", "Glucose Fasting"], "mg/dL", (70, 99), (70, 99), 0, "Glycemic"),
    "Creatinine": (["Serum Creatinine", "Creatinine"], "mg/dL", (0.7, 1.3), (0.6, 1.1), 2, "Renal"),
    "eGFR": (["eGFR", "Estimated GFR"], "mL/min/1.73m2", (90, None), (90, None), 0, "Renal"),
    "Urea": (["Blood Urea", "Urea"], "mg/dL", (17, 43), (17, 43), 0, "Renal"),
    "Total Cholesterol": (["Cholesterol Total", "Total Cholesterol"], "mg/dL", (None, 200), (None, 200), 0, "Lipid"),
    "LDL": (["LDL Cholesterol", "LDL-C"], "mg/dL", (None, 100), (None, 100), 0, "Lipid"),
    "HDL": (["HDL Cholesterol", "HDL-C"], "mg/dL", (40, None), (50, None), 0, "Lipid"),
    "Triglycerides": (["Triglycerides", "TG"], "mg/dL", (None, 150), (None, 150), 0, "Lipid"),
    "Hemoglobin": (["Hemoglobin", "Hb"], "g/dL", (13.0, 17.0), (12.0, 15.0), 1, "Blood count"),
    "MCV": (["MCV", "Mean Corpuscular Volume"], "fL", (80, 100), (80, 100), 0, "Blood count"),
    "WBC": (["Total Leukocyte Count", "WBC Count"], "10^3/uL", (4.0, 11.0), (4.0, 11.0), 1, "Blood count"),
    "Platelets": (["Platelet Count", "Platelets"], "10^3/uL", (150, 410), (150, 410), 0, "Blood count"),
    "Ferritin": (["Serum Ferritin", "Ferritin"], "ng/mL", (30, 400), (15, 150), 0, "Iron"),
    "ALT": (["ALT SGPT", "SGPT"], "U/L", (7, 56), (7, 45), 0, "Liver"),
    "AST": (["AST SGOT", "SGOT"], "U/L", (10, 40), (10, 35), 0, "Liver"),
    "TSH": (["TSH", "Thyroid Stimulating Hormone"], "uIU/mL", (0.4, 4.0), (0.4, 4.0), 2, "Thyroid"),
}


def norm(text: str) -> str:
    return re.sub(r"[^a-z0-9]", "", text.lower())


ALIAS_TO_CANONICAL = {norm(a): canon for canon, spec in TESTS.items() for a in spec[0] + [canon]}


def canonical(name: str):
    """Map a printed test name to a canonical name, or None if we do not know it."""
    return ALIAS_TO_CANONICAL.get(norm(name))

Writing lablens/catalog.py


In [3]:
%%writefile lablens/generate.py
"""Synthetic lab reports in three different layouts (no real patient data)."""
import json
import random
from pathlib import Path

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table

from .catalog import TESTS

DATES = [("2025-03-12", "12-Mar-2025"), ("2025-08-20", "20-Aug-2025"), ("2026-01-15", "15-Jan-2026")]
LABS = ["Sunrise Diagnostics", "CityCare Pathology Lab", "Metro Health Labs"]

# patient -> (age, sex, {test: [value at visit 1, 2, 3]})
PATIENTS = {
    "P001": (52, "M", {"HbA1c": [6.1, 6.6, 7.2], "Fasting Glucose": [108, 121, 138], "Creatinine": [1.0, 1.2, 1.4], "eGFR": [88, 74, 62],
                       "Urea": [30, 36, 44], "Total Cholesterol": [205, 218, 232], "LDL": [128, 140, 152], "HDL": [42, 40, 38],
                       "Triglycerides": [165, 190, 220], "Hemoglobin": [14.8, 14.6, 14.2], "TSH": [2.1, 2.3, 2.2]}),
    "P002": (34, "F", {"Hemoglobin": [11.8, 11.0, 10.2], "MCV": [76, 72, 68], "Ferritin": [18, 12, 8], "WBC": [6.8, 7.1, 6.9],
                       "Platelets": [310, 330, 350], "TSH": [2.8, 3.1, 3.5], "HbA1c": [5.2, 5.2, 5.3]}),
    "P003": (45, "M", {"ALT": [45, 68, 95], "AST": [38, 52, 70], "Triglycerides": [180, 210, 260], "Total Cholesterol": [210, 224, 236],
                       "LDL": [118, 126, 133], "HDL": [38, 36, 35], "Fasting Glucose": [96, 104, 112], "Hemoglobin": [15.1, 15.0, 15.2]}),
    "P004": (29, "F", {"Hemoglobin": [13.4, 13.2, 13.5], "HbA1c": [5.0, 5.1, 5.0], "Total Cholesterol": [168, 172, 165], "LDL": [88, 90, 86],
                       "HDL": [58, 60, 57], "Triglycerides": [98, 105, 92], "TSH": [1.9, 2.0, 1.8], "WBC": [6.2, 6.0, 6.4]}),
}


def ref_text(lo, hi) -> str:
    if lo is not None and hi is not None:
        return f"{lo:g} - {hi:g}"
    return f"< {hi:g}" if lo is None else f"> {lo:g}"


def flag_for(value, lo, hi) -> str:
    if lo is not None and value < lo:
        return "L"
    if hi is not None and value > hi:
        return "H"
    return ""


def make_rows(sex, values):
    rows = []
    for name, value in values.items():
        aliases, unit, ref_m, ref_f, dp, _ = TESTS[name]
        lo, hi = ref_m if sex == "M" else ref_f
        rows.append(dict(canonical=name, printed=aliases[0], value=round(value, dp), unit=unit, ref_low=lo, ref_high=hi,
                         flag=flag_for(value, lo, hi)))
    return rows


def write_pdf(path: Path, patient_id, age, sex, iso, printed_date, rows, layout: str, lab: str):
    styles = getSampleStyleSheet()
    body = styles["BodyText"]
    story = [Paragraph(f"<b>{lab}</b>", styles["Title"]),
             Paragraph(f"Patient ID: {patient_id} &nbsp;&nbsp; Age / Sex: {age} / {sex}", body),
             Paragraph(f"Report Date: {printed_date if layout != 'B' else iso}", body), Spacer(1, 10)]
    if layout == "A":
        table = [["Test", "Result", "Unit", "Reference Range", "Flag"]] + [
            [r["printed"], r["value"], r["unit"], ref_text(r["ref_low"], r["ref_high"]), r["flag"]] for r in rows]
        story.append(Table(table))
    elif layout == "C":
        table = [["Investigation", "Reference Interval", "Observed Value", "Units"]] + [
            [r["printed"], ref_text(r["ref_low"], r["ref_high"]), r["value"], r["unit"]] for r in rows]
        story.append(Table(table))
    else:  # layout B: plain text lines
        for r in rows:
            mark = {"H": " *HIGH*", "L": " *LOW*", "": ""}[r["flag"]]
            story.append(Paragraph(f"{r['printed']} : {r['value']} {r['unit']} ( Ref: {ref_text(r['ref_low'], r['ref_high'])} ){mark}", body))
    SimpleDocTemplate(str(path), pagesize=A4).build(story)


def generate_samples(out_dir="samples", seed=7):
    rng = random.Random(seed)
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    truth = []
    for pid, (age, sex, series) in PATIENTS.items():
        for visit, (iso, printed) in enumerate(DATES):
            values = {t: v[visit] for t, v in series.items()}
            rows = make_rows(sex, values)
            layout = "ABC"[(int(pid[-1]) + visit) % 3]
            lab = LABS[rng.randrange(len(LABS))]
            path = out / f"{pid}_{iso}_layout{layout}.pdf"
            write_pdf(path, pid, age, sex, iso, printed, rows, layout, lab)
            truth.append(dict(file=path.name, patient=pid, date=iso, sex=sex, layout=layout, rows=rows))
    (out / "truth.json").write_text(json.dumps(truth, indent=1))
    return truth

Writing lablens/generate.py


In [4]:
%%writefile lablens/extract.py
"""Read lab-report PDFs into structured rows: test, value, unit, reference range, printed flag."""
import re
from datetime import datetime
from pathlib import Path

import pdfplumber

from .catalog import canonical

NUM = r"\d+(?:\.\d+)?"
RANGE = re.compile(rf"(?P<lo>{NUM})\s*-\s*(?P<hi>{NUM})")
LESS = re.compile(rf"(?:<=?|up to)\s*(?P<hi>{NUM})", re.I)
MORE = re.compile(rf">=?\s*(?P<lo>{NUM})")
FLAG = re.compile(r"\s\*?(?:HIGH|LOW|H|L)\*?\s*$")
SKIP = ("patient", "age", "report", "collected", "lab", "sample", "page", "test ", "investigation")
DATE_FORMATS = ("%d-%b-%Y", "%Y-%m-%d", "%d/%m/%Y", "%d %b %Y")


def read_text(pdf_path) -> str:
    with pdfplumber.open(pdf_path) as pdf:
        return "\n".join((page.extract_text() or "") for page in pdf.pages)


def parse_header(text: str) -> dict:
    head = {"date": None, "sex": None, "age": None, "patient": None}
    m = re.search(r"Report Date\s*[:\-]?\s*(\S+)", text)
    if m:
        for fmt in DATE_FORMATS:
            try:
                head["date"] = datetime.strptime(m.group(1), fmt).date().isoformat()
                break
            except ValueError:
                continue
    m = re.search(r"Age\s*/\s*Sex\s*[:\-]?\s*(\d+)\s*/\s*([MF])", text)
    if m:
        head["age"], head["sex"] = int(m.group(1)), m.group(2)
    m = re.search(r"Patient ID\s*[:\-]?\s*(\S+)", text)
    if m:
        head["patient"] = m.group(1)
    return head


def parse_line(line: str):
    """Return a row dict for one text line, or None if the line is not a result row."""
    line = line.strip()
    if not line or line.lower().startswith(SKIP):
        return None
    flag_m = FLAG.search(" " + line)
    flag = flag_m.group(0).strip().strip("*")[0] if flag_m else ""
    line = FLAG.sub("", " " + line).strip()
    line = re.sub(r"[()\[\]:]|\bRef\b", " ", line)
    lo = hi = None
    ref = RANGE.search(line)
    if ref:
        lo, hi = float(ref["lo"]), float(ref["hi"])
    else:
        less, more = LESS.search(line), MORE.search(line)
        if less:
            hi, ref = float(less["hi"]), less
        elif more:
            lo, ref = float(more["lo"]), more
    if ref is None:
        return None
    line = (line[:ref.start()] + " " + line[ref.end():]).split()
    for i, token in enumerate(line):
        try:
            value = float(token)
        except ValueError:
            continue
        if i == 0 or i + 1 >= len(line):
            return None
        name, unit = " ".join(line[:i]), line[i + 1]
        return dict(printed=name, canonical=canonical(name), value=value, unit=unit, ref_low=lo, ref_high=hi, flag=flag)
    return None


def extract_report(pdf_path) -> dict:
    text = read_text(pdf_path)
    rows = [r for r in (parse_line(l) for l in text.splitlines()) if r]
    return {"file": Path(pdf_path).name, **parse_header(text), "rows": rows}


def score_extraction(truth: list, folder) -> dict:
    """Field-level accuracy of the extractor against the known values of the synthetic reports."""
    fields = ok = missing = 0
    per_layout: dict = {}
    for rec in truth:
        got = {r["canonical"]: r for r in extract_report(Path(folder) / rec["file"])["rows"]}
        for row in rec["rows"]:
            g = got.get(row["canonical"])
            layout = per_layout.setdefault(rec["layout"], [0, 0])
            for key in ("value", "unit", "ref_low", "ref_high"):
                fields += 1
                layout[1] += 1
                if g is None:
                    missing += 1
                elif g[key] == row[key]:
                    ok += 1
                    layout[0] += 1
    return {"fields": fields, "correct": ok, "accuracy": round(ok / fields, 4), "missing_field_count": missing,
            "by_layout": {k: f"{a}/{b}" for k, (a, b) in sorted(per_layout.items())}}

Writing lablens/extract.py


In [5]:
import sys, pandas as pd
sys.path.insert(0, "/content")
from lablens.generate import generate_samples
from lablens.extract import extract_report, score_extraction

truth = generate_samples("samples")
print(len(truth), "synthetic reports written to samples/")
sample = extract_report("samples/P001_2025-03-12_layoutB.pdf")
print({k: sample[k] for k in ("patient", "date", "age", "sex")})
display(pd.DataFrame(sample["rows"]))
print(score_extraction(truth, "samples"))

12 synthetic reports written to samples/
{'patient': 'P001', 'date': '2025-03-12', 'age': 52, 'sex': 'M'}


,printed,canonical,value,unit,ref_low,ref_high,flag
0,Hemoglobin A1c,HbA1c,6.1,%,4.0,5.6,H
1,Fasting Blood Sugar,Fasting Glucose,108.0,mg/dL,70.0,99.0,H
2,Serum Creatinine,Creatinine,1.0,mg/dL,0.7,1.3,
3,eGFR,eGFR,88.0,mL/min/1.73m2,90.0,NaN,L
4,Blood Urea,Urea,30.0,mg/dL,17.0,43.0,
5,Cholesterol Total,Total Cholesterol,205.0,mg/dL,NaN,200.0,H
6,LDL Cholesterol,LDL,128.0,mg/dL,NaN,100.0,H
7,HDL Cholesterol,HDL,42.0,mg/dL,40.0,NaN,
8,Triglycerides,Triglycerides,165.0,mg/dL,NaN,150.0,H
9,Hemoglobin,Hemoglobin,14.8,g/dL,13.0,17.0,


{'fields': 408, 'correct': 408, 'accuracy': 1.0, 'missing_field_count': 0, 'by_layout': {'A': '136/136', 'B': '136/136', 'C': '136/136'}}


In [6]:
%%writefile lablens/analyze.py
"""Trend analysis and doctor-ready summaries built ONLY from the extracted values (no LLM, no invented numbers).

Decision support for a clinician to review. It does not diagnose.
"""
import pandas as pd

from .catalog import TESTS


def nn(x):
    """NaN -> None (pandas stores open-ended reference limits as NaN)."""
    return None if x is None or pd.isna(x) else float(x)


def fmt_ref(lo, hi) -> str:
    lo, hi = nn(lo), nn(hi)
    if lo is not None and hi is not None:
        return f"{lo:g}-{hi:g}"
    return f"<{hi:g}" if lo is None else f">{lo:g}"


DISCLAIMER = "Decision support generated from the extracted values only. Not a diagnosis. The clinician must review."


def status(v, lo, hi) -> str:
    if lo is not None and v < lo:
        return "low"
    if hi is not None and v > hi:
        return "high"
    return "normal"


def outside(v, lo, hi) -> float:
    """How far outside the reference range a value is, as a fraction of the nearest limit (0 = inside)."""
    if lo is not None and v < lo:
        return (lo - v) / abs(lo)
    if hi is not None and v > hi:
        return (v - hi) / abs(hi)
    return 0.0


def build_history(reports: list) -> pd.DataFrame:
    rows = [dict(date=rep["date"], test=r["canonical"], value=r["value"], unit=r["unit"], ref_low=r["ref_low"], ref_high=r["ref_high"],
                 status=status(r["value"], r["ref_low"], r["ref_high"]), group=TESTS[r["canonical"]][5])
            for rep in reports for r in rep["rows"] if r["canonical"]]
    return pd.DataFrame(rows).sort_values(["test", "date"]).reset_index(drop=True)


def approaching(v, lo, hi, direction) -> str:
    """Inside the range but close to a limit and moving toward it."""
    if hi is not None and direction == "rising" and ((lo is not None and (v - lo) / (hi - lo) > 0.85) or (lo is None and v >= 0.9 * hi)):
        return "upper limit"
    if lo is not None and direction == "falling" and ((hi is not None and (v - lo) / (hi - lo) < 0.15) or (hi is None and v <= 1.1 * lo)):
        return "lower limit"
    return ""


def trend_table(hist: pd.DataFrame) -> pd.DataFrame:
    out = []
    for test, g in hist.groupby("test"):
        if len(g) < 2:
            continue
        first, last = g.iloc[0], g.iloc[-1]
        pct = (last.value - first.value) / abs(first.value) * 100 if first.value else 0.0
        direction = "stable" if abs(pct) < 5 else ("rising" if pct > 0 else "falling")
        lo, hi = nn(last.ref_low), nn(last.ref_high)
        d0, d1 = outside(first.value, nn(first.ref_low), nn(first.ref_high)), outside(last.value, lo, hi)
        near = approaching(last.value, lo, hi, direction) if d1 == 0 else ""
        movement = "worsening" if d1 > d0 + 0.02 else "improving" if d1 < d0 - 0.02 else ("approaching " + near if near else "stable")
        out.append(dict(test=test, group=last.group, unit=last.unit, values=" \u2192 ".join(f"{v:g}" for v in g.value), first=first.value,
                        last=last.value, change_pct=round(pct, 1), direction=direction, latest_status=last.status, movement=movement,
                        ref_low=last.ref_low, ref_high=last.ref_high))
    return pd.DataFrame(out)


def patterns(trends: pd.DataFrame) -> list:
    """Cross-panel patterns: (title, why, follow-up tests to consider, what to discuss)."""
    t = {r.test: r for r in trends.itertuples()}
    high = lambda n: n in t and t[n].latest_status == "high"          # noqa: E731
    low = lambda n: n in t and t[n].latest_status == "low"            # noqa: E731
    moving = lambda n, d: n in t and t[n].direction == d              # noqa: E731
    found = []
    glyc = [n for n in ("HbA1c", "Fasting Glucose") if high(n) and moving(n, "rising")]
    renal = (moving("Creatinine", "rising") and t["Creatinine"].movement != "stable") or (moving("eGFR", "falling") and t["eGFR"].movement != "stable") \
        if ("Creatinine" in t or "eGFR" in t) else False
    if glyc and renal:
        found.append(("Rising glucose markers with declining kidney-function markers",
                      f"{' and '.join(glyc)} above range and rising, while creatinine/eGFR are worsening across visits.",
                      ["Urine albumin-to-creatinine ratio", "Repeat creatinine and eGFR", "Repeat HbA1c in about 3 months"],
                      ["Glycemic control and medication adherence", "Kidney-safe prescribing"]))
    if (high("LDL") or high("Total Cholesterol")) and (glyc or high("Triglycerides")):
        found.append(("Lipid markers and glucose/triglycerides above range together",
                      "LDL or total cholesterol is high alongside high triglycerides or rising glucose markers.",
                      ["Fasting lipid profile repeat", "Blood pressure and cardiovascular risk assessment"],
                      ["Diet, activity and weight", "Whether lipid-lowering treatment is under review"]))
    if low("Hemoglobin") and low("MCV"):
        extra = " Ferritin is also low." if low("Ferritin") else ""
        found.append(("Low hemoglobin with low MCV (microcytic pattern)",
                      "Hemoglobin and MCV are both below range and trending down." + extra,
                      ["Iron studies (serum iron, TIBC, transferrin saturation)", "Peripheral blood smear", "Repeat complete blood count"],
                      ["Diet, blood-loss history and menstrual history", "Symptoms of anaemia"]))
    if high("ALT") and high("AST") and moving("ALT", "rising"):
        found.append(("Liver enzymes above range and rising",
                      "ALT and AST are both above range, with ALT rising across visits." + (" Triglycerides are also high." if high("Triglycerides") else ""),
                      ["Repeat liver panel", "Bilirubin and GGT", "Imaging as clinically indicated"],
                      ["Medicines, supplements and alcohol history", "Weight and metabolic risk"]))
    return found


def summarise(patient: dict, hist: pd.DataFrame, trends: pd.DataFrame, pats: list) -> str:
    dates = sorted(hist.date.unique())
    latest = hist[hist.date == dates[-1]]
    lines = [f"Patient {patient.get('patient')} ({patient.get('age')} {patient.get('sex')}): {len(dates)} reports, {dates[0]} to {dates[-1]}."]
    flagged = latest[latest.status != "normal"]
    lines.append("Latest results outside the reference range: " + (
        "; ".join(f"{r.test} {r.value:g} {r.unit} ({r.status}, ref {fmt_ref(r.ref_low, r.ref_high)})"
                  for r in flagged.itertuples()) or "none") + ".")
    worse = trends[trends.movement == "worsening"]
    if len(worse):
        lines.append("Worsening across visits: " + "; ".join(f"{r.test} {r.values} {r.unit} ({r.change_pct:+g}%)" for r in worse.itertuples()) + ".")
    better = trends[trends.movement == "improving"]
    if len(better):
        lines.append("Improving: " + "; ".join(f"{r.test} {r.values} {r.unit}" for r in better.itertuples()) + ".")
    near = trends[trends.movement.str.startswith("approaching")]
    if len(near):
        lines.append("Within range but moving toward a limit: " + "; ".join(f"{r.test} {r.values} {r.unit}" for r in near.itertuples()) + ".")
    for title, why, tests, talk in pats:
        lines.append(f"Pattern for clinician review: {title}. {why}")
    if pats:
        lines.append("Follow-up tests to consider: " + "; ".join(dict.fromkeys(x for p in pats for x in p[2])) + ".")
        lines.append("Discussion points: " + "; ".join(dict.fromkeys(x for p in pats for x in p[3])) + ".")
    watch = list(worse.test) + list(near.test)
    lines.append("Monitor next: " + (", ".join(dict.fromkeys(watch)) if watch else "no marker is trending toward a limit") + ".")
    lines.append(DISCLAIMER)
    return "\n".join(lines)


def analyse_patient(reports: list) -> dict:
    reports = sorted(reports, key=lambda r: r["date"])
    hist = build_history(reports)
    trends = trend_table(hist)
    pats = patterns(trends) if len(trends) else []
    meta = {k: reports[-1].get(k) for k in ("patient", "age", "sex")}
    return dict(history=hist, trends=trends, patterns=pats, summary=summarise(meta, hist, trends, pats), meta=meta)

Writing lablens/analyze.py


In [7]:
import glob, collections, importlib
import lablens.analyze as az
importlib.reload(az)
from lablens.extract import extract_report

by = collections.defaultdict(list)
for f in sorted(glob.glob("samples/*.pdf")):
    r = extract_report(f)
    by[r["patient"]].append(r)
results = {pid: az.analyse_patient(reps) for pid, reps in by.items()}

for pid in ["P001", "P002"]:
    print("=====", pid); print(results[pid]["summary"]); print()
display(results["P001"]["trends"][["test", "values", "change_pct", "latest_status", "movement"]])

# self-checks on the synthetic patients (rules and data were both written by me)
names = {pid: [p[0] for p in r["patterns"]] for pid, r in results.items()}
assert any("kidney" in n for n in names["P001"]) and any("microcytic" in n for n in names["P002"])
assert any("Liver" in n for n in names["P003"]) and names["P004"] == []
assert (results["P004"]["trends"].movement == "stable").all()
print("self-checks passed:", {pid: len(n) for pid, n in names.items()}, "patterns found")

===== P001
Patient P001 (52 M): 3 reports, 2025-03-12 to 2026-01-15.
Latest results outside the reference range: Creatinine 1.4 mg/dL (high, ref 0.7-1.3); Fasting Glucose 138 mg/dL (high, ref 70-99); HDL 38 mg/dL (low, ref >40); HbA1c 7.2 % (high, ref 4-5.6); LDL 152 mg/dL (high, ref <100); Total Cholesterol 232 mg/dL (high, ref <200); Triglycerides 220 mg/dL (high, ref <150); Urea 44 mg/dL (high, ref 17-43); eGFR 62 mL/min/1.73m2 (low, ref >90).
Worsening across visits: Creatinine 1 → 1.2 → 1.4 mg/dL (+40%); Fasting Glucose 108 → 121 → 138 mg/dL (+27.8%); HDL 42 → 40 → 38 mg/dL (-9.5%); HbA1c 6.1 → 6.6 → 7.2 % (+18%); LDL 128 → 140 → 152 mg/dL (+18.8%); Total Cholesterol 205 → 218 → 232 mg/dL (+13.2%); Triglycerides 165 → 190 → 220 mg/dL (+33.3%); Urea 30 → 36 → 44 mg/dL (+46.7%); eGFR 88 → 74 → 62 mL/min/1.73m2 (-29.5%).
Pattern for clinician review: Rising glucose markers with declining kidney-function markers. HbA1c and Fasting Glucose above range and rising, while creatinine/eGFR 

,test,values,change_pct,latest_status,movement
0,Creatinine,1 → 1.2 → 1.4,40.0,high,worsening
1,Fasting Glucose,108 → 121 → 138,27.8,high,worsening
2,HDL,42 → 40 → 38,-9.5,low,worsening
3,HbA1c,6.1 → 6.6 → 7.2,18.0,high,worsening
4,Hemoglobin,14.8 → 14.6 → 14.2,-4.1,normal,stable
5,LDL,128 → 140 → 152,18.8,high,worsening
6,TSH,2.1 → 2.3 → 2.2,4.8,normal,stable
7,Total Cholesterol,205 → 218 → 232,13.2,high,worsening
8,Triglycerides,165 → 190 → 220,33.3,high,worsening
9,Urea,30 → 36 → 44,46.7,high,worsening


self-checks passed: {'P001': 2, 'P002': 1, 'P003': 2, 'P004': 0} patterns found
